# 02 DistilBERT Baseline 2 and Ablation B

Upload `06_model_ready.zip` to `/content/`. Optionally upload `tfidf_logreg_outputs.zip` for combined comparison. Run all cells, then download `/content/distilbert_baselines_outputs.zip`.

## Dependency Check

In [ ]:
import importlib.util, subprocess, sys
required = {
    "pandas": "pandas", "numpy": "numpy", "sklearn": "scikit-learn", "matplotlib": "matplotlib",
    "joblib": "joblib", "tqdm": "tqdm", "tabulate": "tabulate", "transformers": "transformers", "torch": "torch"
}
missing = [pip for mod, pip in required.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("Dependency check complete. Installed:", missing)


## Paths, Data Loading, and Output Folders

In [ ]:
from pathlib import Path
import zipfile, shutil, json, random, datetime, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

TEXT_COL = "model_text"
LABEL_COL = "label_id"
ID_COL = "final_row_id"
SEEDS = [42, 7, 123]
RUN_SEEDS = [42, 7, 123]  # Temporarily set to [42] for faster smoke runs.
THRESHOLD = 0.5

CONTENT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
DATA_ZIP = CONTENT_DIR / "06_model_ready.zip"
if DATA_ZIP.exists():
    print(f"Found dataset ZIP, extracting: {DATA_ZIP}")
    with zipfile.ZipFile(DATA_ZIP, "r") as zf:
        zf.extractall(CONTENT_DIR)
else:
    print("No /content/06_model_ready.zip found. Looking for extracted dataset.")

def find_data_dir():
    candidates = [
        CONTENT_DIR / "06_model_ready",
        CONTENT_DIR / "model_ready",
        CONTENT_DIR / "data" / "06_model_ready",
        CONTENT_DIR / "thesis-modeling" / "data" / "06_model_ready",
        CONTENT_DIR / "ralf revision and GA model" / "data" / "06_model_ready",
        Path.cwd() / "06_model_ready",
        Path.cwd() / "model_ready",
        Path.cwd() / "data" / "06_model_ready",
    ]
    for base in [CONTENT_DIR, Path.cwd()]:
        candidates.extend([p for p in base.rglob("06_model_ready") if p.is_dir()])
    seen = []
    for p in candidates:
        if p not in seen:
            seen.append(p)
    for p in seen:
        if (p / "clean" / "train_clean.csv").exists():
            return p.resolve()
    raise FileNotFoundError("Could not locate 06_model_ready. Upload 06_model_ready.zip to /content and rerun.")

DATA_DIR = find_data_dir()
BASE_DIR = CONTENT_DIR / "baseline_modeling_outputs"
BASE_DIR.mkdir(parents=True, exist_ok=True)
TRAINED_DIR = BASE_DIR / "trained_models"
RESULTS_DIR = BASE_DIR / "results"
REPORTS_DIR = BASE_DIR / "reports"
METRICS_DIR = RESULTS_DIR / "metrics"
PRED_DIR = RESULTS_DIR / "predictions"
FIGURE_DIR = RESULTS_DIR / "figures"
DEGRADATION_DIR = RESULTS_DIR / "degradation_tables"
for d in [TRAINED_DIR, RESULTS_DIR, REPORTS_DIR, METRICS_DIR, PRED_DIR, FIGURE_DIR, DEGRADATION_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SPLIT_PATHS = {
    "train_clean": DATA_DIR / "clean" / "train_clean.csv",
    "val_clean": DATA_DIR / "clean" / "val_clean.csv",
    "test_clean": DATA_DIR / "clean" / "test_clean.csv",
    "train_augmented_ablation_b": DATA_DIR / "augmented_training" / "train_augmented_for_ablation_b.csv",
    "test_adv_10": DATA_DIR / "adversarial_test" / "test_adv_10.csv",
    "test_adv_20": DATA_DIR / "adversarial_test" / "test_adv_20.csv",
    "test_adv_30": DATA_DIR / "adversarial_test" / "test_adv_30.csv",
}
EVAL_SPLITS = ["test_clean", "test_adv_10", "test_adv_20", "test_adv_30"]
for name, path in SPLIT_PATHS.items():
    if name != "train_augmented_ablation_b" and not path.exists():
        raise FileNotFoundError(f"Missing required split {name}: {path}")
print("DATA_DIR =", DATA_DIR)
print("BASE_DIR =", BASE_DIR)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)

def load_split(split_name):
    path = SPLIT_PATHS[split_name]
    if not path.exists():
        raise FileNotFoundError(f"Missing split file: {path}")
    df = pd.read_csv(path)
    assert TEXT_COL in df.columns, f"{path} missing {TEXT_COL}"
    assert LABEL_COL in df.columns, f"{path} missing {LABEL_COL}"
    df[TEXT_COL] = df[TEXT_COL].fillna("").astype(str)
    df[LABEL_COL] = df[LABEL_COL].astype(int)
    if ID_COL not in df.columns:
        df[ID_COL] = np.arange(len(df))
    return df

train_clean_df = load_split("train_clean")
val_clean_df = load_split("val_clean")
test_dfs = {split: load_split(split) for split in EVAL_SPLITS}
print("Loaded rows:", {"train_clean": len(train_clean_df), "val_clean": len(val_clean_df), **{k: len(v) for k, v in test_dfs.items()}})


## Evaluation Helpers

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

def binary_metrics(y_true, prob, model_name, split, seed, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob)
    pred = (prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(y_true, pred, pos_label=1, average="binary", zero_division=0)
    return {
        "model": model_name, "split": split, "seed": seed, "n_rows": int(len(y_true)),
        "accuracy": accuracy_score(y_true, pred), "precision_smishing": precision,
        "recall_smishing": recall, "f1_smishing": f1,
        "false_negative_rate": fn / (fn + tp) if (fn + tp) else 0.0,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else 0.0,
        "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
        "support_ham": int((y_true == 0).sum()), "support_smishing": int((y_true == 1).sum()),
        "threshold": threshold,
    }

def save_confusion(m, out_path, title):
    mat = np.array([[m["tn"], m["fp"]], [m["fn"], m["tp"]]])
    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(mat, cmap="Blues")
    ax.set_xticks([0, 1], ["Pred Ham", "Pred Smishing"])
    ax.set_yticks([0, 1], ["True Ham", "True Smishing"])
    ax.set_title(title)
    for i in range(2):
        for j in range(2): ax.text(j, i, str(mat[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout(); fig.savefig(out_path, dpi=160); plt.close(fig)

def degradation_table(metrics_df, model_name):
    def get(split, col):
        row = metrics_df[metrics_df["split"] == split]
        return np.nan if row.empty else float(row.iloc[0][col])
    row = {
        "model": model_name,
        "clean_recall": get("test_clean", "recall_smishing"), "adv10_recall": get("test_adv_10", "recall_smishing"),
        "adv20_recall": get("test_adv_20", "recall_smishing"), "adv30_recall": get("test_adv_30", "recall_smishing"),
        "clean_f1": get("test_clean", "f1_smishing"), "adv10_f1": get("test_adv_10", "f1_smishing"),
        "adv20_f1": get("test_adv_20", "f1_smishing"), "adv30_f1": get("test_adv_30", "f1_smishing"),
        "clean_fnr": get("test_clean", "false_negative_rate"), "adv30_fnr": get("test_adv_30", "false_negative_rate"),
    }
    row["clean_to_adv30_recall_drop"] = row["clean_recall"] - row["adv30_recall"]
    row["clean_to_adv30_f1_drop"] = row["clean_f1"] - row["adv30_f1"]
    row["clean_to_adv30_fnr_increase"] = row["adv30_fnr"] - row["clean_fnr"]
    return pd.DataFrame([row])

def mean_std_metrics(df, model_name):
    metric_cols = ["accuracy", "precision_smishing", "recall_smishing", "f1_smishing", "false_negative_rate", "false_positive_rate"]
    out = df.groupby("split")[metric_cols].agg(["mean", "std"]).reset_index()
    out.columns = ["_".join([x for x in col if x]) for col in out.columns.to_flat_index()]
    out.insert(0, "model", model_name)
    return out


## Train and Evaluate DistilBERT Models

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = "distilbert-base-cased"
MAX_LENGTH = 128
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
MAX_EPOCHS = 10
PATIENCE = 3
WARMUP_RATIO = 0.10
GRAD_CLIP_NORM = 1.0
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE =", DEVICE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
train_aug_df = load_split("train_augmented_ablation_b")

class SmsDataset(Dataset):
    def __init__(self, df):
        self.texts = df[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = df[LABEL_COL].astype(int).to_numpy()
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], truncation=True, padding="max_length", max_length=MAX_LENGTH, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def class_weights_tensor(labels):
    classes = np.array([0, 1])
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=np.asarray(labels).astype(int))
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def train_distilbert(model_name_key, train_df, seed):
    set_seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    model_dir = TRAINED_DIR / model_name_key / f"seed{seed}"
    model_dir.mkdir(parents=True, exist_ok=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    train_loader = DataLoader(SmsDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(SmsDataset(val_clean_df), batch_size=BATCH_SIZE, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = MAX_EPOCHS * len(train_loader)
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    weights = class_weights_tensor(train_df[LABEL_COL])
    loss_fn = torch.nn.CrossEntropyLoss(weight=weights)
    best_state, best_val_loss, best_epoch, stale, history = None, float("inf"), 0, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train(); total_loss = 0.0; train_probs=[]; train_y=[]
        for batch in tqdm(train_loader, desc=f"{model_name_key} seed {seed} epoch {epoch}"):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            loss = loss_fn(out.logits, batch["labels"])
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            optimizer.step(); scheduler.step()
            total_loss += loss.item() * len(batch["labels"])
            train_probs.extend(torch.softmax(out.logits.detach(), dim=1)[:,1].cpu().numpy())
            train_y.extend(batch["labels"].detach().cpu().numpy())
        train_loss = total_loss / len(train_df)
        model.eval(); val_loss_total=0.0; val_probs=[]; val_y=[]
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
                loss = loss_fn(out.logits, batch["labels"])
                val_loss_total += loss.item() * len(batch["labels"])
                val_probs.extend(torch.softmax(out.logits, dim=1)[:,1].cpu().numpy())
                val_y.extend(batch["labels"].cpu().numpy())
        val_loss = val_loss_total / len(val_clean_df)
        tm = binary_metrics(train_y, train_probs, model_name_key, "train", seed)
        vm = binary_metrics(val_y, val_probs, model_name_key, "val_clean", seed)
        history.append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "train_recall_smishing": tm["recall_smishing"], "train_f1_smishing": tm["f1_smishing"], "val_recall_smishing": vm["recall_smishing"], "val_f1_smishing": vm["f1_smishing"]})
        if val_loss < best_val_loss - 1e-5:
            best_val_loss, best_epoch, stale = val_loss, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k,v in model.state_dict().items()}
            model.save_pretrained(model_dir / "best_model")
            tokenizer.save_pretrained(model_dir / "best_model")
        else:
            stale += 1
            if stale >= PATIENCE: break
    model.load_state_dict(best_state)
    hist = pd.DataFrame(history)
    hist.to_csv(METRICS_DIR / f"{model_name_key}_seed{seed}_training_history.csv", index=False)
    fig, ax = plt.subplots(figsize=(7,4)); ax.plot(hist["epoch"], hist["train_loss"], label="train loss"); ax.plot(hist["epoch"], hist["val_loss"], label="val loss"); ax.set_title(f"{model_name_key} seed {seed}"); ax.set_xlabel("Epoch"); ax.set_ylabel("Loss"); ax.legend(); fig.tight_layout(); fig.savefig(FIGURE_DIR / f"{model_name_key}_training_curve_seed{seed}.png", dpi=160); plt.close(fig)
    (model_dir / "training_config.json").write_text(json.dumps({"model_name": MODEL_NAME, "seed": seed, "best_epoch": best_epoch, "text_col": TEXT_COL, "label_col": LABEL_COL, "class_weights": weights.detach().cpu().tolist(), "validation_split": "val_clean", "threshold": THRESHOLD}, indent=2), encoding="utf-8")
    return model, hist

@torch.no_grad()
def predict_distilbert(model, df):
    loader = DataLoader(SmsDataset(df), batch_size=BATCH_SIZE, shuffle=False)
    model.eval(); probs=[]
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
        probs.extend(torch.softmax(out.logits, dim=1)[:,1].cpu().numpy())
    return np.asarray(probs)

def run_distilbert_experiment(model_name_key, train_df):
    rows=[]
    for seed in RUN_SEEDS:
        print(f"Training {model_name_key} seed={seed}")
        model, hist = train_distilbert(model_name_key, train_df, seed)
        for split, df in test_dfs.items():
            prob = predict_distilbert(model, df)
            m = binary_metrics(df[LABEL_COL], prob, model_name_key, split, seed)
            rows.append(m)
            pred = (prob >= THRESHOLD).astype(int)
            pd.DataFrame({ID_COL: df[ID_COL], TEXT_COL: df[TEXT_COL], "true_label": df[LABEL_COL], "predicted_label": pred, "predicted_probability": prob}).to_csv(PRED_DIR / f"{model_name_key}_seed{seed}_predictions_{split}.csv", index=False)
            save_confusion(m, FIGURE_DIR / f"{model_name_key}_confusion_matrix_seed{seed}_{split}.png", f"{model_name_key} seed {seed} {split}")
    metrics = pd.DataFrame(rows)
    metrics.to_csv(METRICS_DIR / f"{model_name_key}_metrics_by_seed.csv", index=False)
    mean_std = mean_std_metrics(metrics, model_name_key)
    mean_std.to_csv(METRICS_DIR / f"{model_name_key}_metrics_mean_std.csv", index=False)
    mean_for_deg = metrics.groupby("split", as_index=False)[["accuracy", "precision_smishing", "recall_smishing", "f1_smishing", "false_negative_rate", "false_positive_rate"]].mean(numeric_only=True)
    degradation_table(mean_for_deg, model_name_key).to_csv(DEGRADATION_DIR / f"{model_name_key}_degradation_table.csv", index=False)
    return metrics, mean_std

baseline2_metrics, baseline2_mean_std = run_distilbert_experiment("distilbert_baseline2", train_clean_df)
ablation_b_metrics, ablation_b_mean_std = run_distilbert_experiment("distilbert_ablation_b", train_aug_df)


## Reports, Optional Comparison, and Output ZIP

In [ ]:
def write_distilbert_summary(model_name_key, train_desc, mean_std):
    deg = pd.read_csv(DEGRADATION_DIR / f"{model_name_key}_degradation_table.csv")
    (REPORTS_DIR / f"{model_name_key}_summary.md").write_text(
        f"# {model_name_key} Summary\n\n"
        f"- Training data: {train_desc}.\n"
        "- Validation data: val_clean.csv only for validation/early stopping.\n"
        "- Test data: held out for final evaluation only.\n"
        "- Architecture: AutoModelForSequenceClassification(num_labels=2), equivalent binary classification using smishing as positive class probability.\n"
        "- Class weights are used in CrossEntropyLoss; clean training is balanced, while augmented training may be smishing-heavy.\n\n"
        "## Mean/Std Metrics\n\n" + mean_std.to_markdown(index=False) + "\n\n"
        "## Degradation\n\n" + deg.to_markdown(index=False) + "\n",
        encoding="utf-8")
write_distilbert_summary("distilbert_baseline2", "clean/train_clean.csv only", baseline2_mean_std)
write_distilbert_summary("distilbert_ablation_b", "augmented_training/train_augmented_for_ablation_b.csv only", ablation_b_mean_std)

# Optional combined comparison with TF-IDF if uploaded.
tfidf_zip = CONTENT_DIR / "tfidf_logreg_outputs.zip"
if tfidf_zip.exists():
    print("Found TF-IDF output ZIP; extracting for optional comparison.")
    with zipfile.ZipFile(tfidf_zip, "r") as zf: zf.extractall(BASE_DIR)
else:
    print("TF-IDF outputs unavailable; combined comparison will include DistilBERT models only unless TF-IDF metrics already exist.")
frames=[]; missing=[]
for label, path in [
    ("TF-IDF Baseline 1", METRICS_DIR / "tfidf_baseline1_metrics_mean_std.csv"),
    ("TF-IDF Ablation A", METRICS_DIR / "tfidf_ablation_a_metrics_mean_std.csv"),
    ("DistilBERT Baseline 2", METRICS_DIR / "distilbert_baseline2_metrics_mean_std.csv"),
    ("DistilBERT Ablation B", METRICS_DIR / "distilbert_ablation_b_metrics_mean_std.csv"),
]:
    if path.exists():
        df=pd.read_csv(path); df.insert(0,"display_name",label); frames.append(df)
    else: missing.append(label)
if frames:
    comp=pd.concat(frames, ignore_index=True, sort=False)
    comp.to_csv(METRICS_DIR / "baselines_ablation_final_comparison.csv", index=False)
    (REPORTS_DIR / "baselines_ablation_final_comparison.md").write_text("# Baselines and Ablations Final Comparison\n\n" + comp.to_markdown(index=False) + "\n\nMissing outputs: " + (", ".join(missing) if missing else "none") + ".\n", encoding="utf-8")

(REPORTS_DIR / "distilbert_baselines_audit_summary.md").write_text(
    "# DistilBERT Baselines Audit Summary\n\n"
    "- Did Baseline 2 use only train_clean.csv for training? Yes.\n"
    "- Did Ablation B use only train_augmented_for_ablation_b.csv for training? Yes.\n"
    "- Was val_clean.csv used only for validation/early stopping? Yes.\n"
    "- Were test sets used only for final evaluation? Yes.\n"
    "- Are there signs of overfitting from train/val loss curves? Inspect per-seed training curves and histories in results/.\n"
    "- Why might Ablation B perform strongly on adversarial sets? It is trained on adversarially augmented text and may learn perturbation-invariant or perturbation-specific cues.\n"
    "- Did adversarial performance degrade reasonably from clean to adv30? See degradation tables.\n"
    "- Are adversarial perturbations too easy or label cues? Review adversarial gains, false negatives, and whether perturbation artifacts become predictive.\n",
    encoding="utf-8")

zip_path = CONTENT_DIR / "distilbert_baselines_outputs.zip"
if zip_path.exists(): zip_path.unlink()
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for rel in ["trained_models/distilbert_baseline2", "trained_models/distilbert_ablation_b", "results/metrics", "results/predictions", "results/figures", "results/degradation_tables", "reports"]:
        src=BASE_DIR/rel
        if src.exists():
            for file in src.rglob("*"):
                if file.is_file(): zf.write(file, file.relative_to(BASE_DIR))
        else: print("Missing for ZIP:", src)
print(f"? DistilBERT output ZIP created: {zip_path}")
required=["trained_models/distilbert_baseline2", "trained_models/distilbert_ablation_b", "results/metrics", "results/predictions", "results/figures", "results/degradation_tables", "reports"]
with zipfile.ZipFile(zip_path,"r") as zf:
    names=set(zf.namelist())
    for entry in required:
        ok=any(n.startswith(entry.rstrip('/') + '/') for n in names)
        print(f"{entry}: {'FOUND' if ok else 'MISSING'}")
print("Download from Colab sidebar:", zip_path)
